## Introduction

Malaria is a life-threatening disease caused by parasites transmitted through the bites of infected female Anopheles mosquitoes. Early detection of malaria-infected cells plays a crucial role in preventing the disease from advancing and aiding faster treatment. In this project, the goal is to build a Convolutional Neural Network (CNN) model capable of distinguishing between parasitized and uninfected red blood cells based on microscopic images.
The dataset used for this project is publicly available and contains cell images labeled as either "Parasitized" or "Uninfected." The model is trained to identify subtle patterns and features that may not be easily noticeable to the human eye, allowing for fast and accurate classification. This approach aims to contribute to the automation of malaria detection processes in low-resource regions where expert diagnosis may not be readily available.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# The dataset was randomly split into training (80%) and validation (20%) using ImageDataGenerator.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_path = "/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images"

img_size=224
batch_size=32

datagen= ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator= datagen.flow_from_directory(
    base_path,
    target_size=(img_size, img_size),
    class_mode="binary",
    subset="training",
    shuffle=True,
    seed=42
)

val_generator=datagen.flow_from_directory(
    base_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode="binary",
    subset="validation",
    shuffle=True,
    seed=42
)

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(train_generator)

plt.figure(figsize=(12,6))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(images[i])
    plt.title("Parasitized" if labels[i] == 1 else "Uninfected")
    plt.axis('off')
plt.tight_layout()
plt.show()


#from chatgpt

In [ ]:
#A neural network model with convolutional layers was implemented to learn features from the input images and perform classification.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model=Sequential()
model.add(Conv2D(32, (3,3), activation="relu", input_shape=(img_size,img_size,3)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, (3,3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dropout(0.6))
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.6))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
history=model.fit(train_generator, epochs=5, validation_data=val_generator)

In [ ]:
#saving model

model.save('/kaggle/working/malaria_cnn_model2.h5')


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Model Accuracy Graph')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#testing

images, labels = next(val_generator)

fig, axs = plt.subplots(2, 3, figsize=(12, 6))
for ax, img, label in zip(axs.flat, images[:6], labels[:6]):
    pred = model.predict(img[np.newaxis, ...])[0][0]
    ax.imshow(img)
    ax.set_title(f"T: {'P' if pred>0.5 else 'U'} | G: {'P' if label==1 else 'U'}")
    ax.axis('off')
plt.tight_layout()
plt.show()


#from chatgpt

# Transfer Learning

VGG16 was imported with ImageNet weights and `include_top=False` to use its convolutional base for feature extraction. Custom dense layers were added on top for classification.


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, Input
from tensorflow.keras.optimizers import Adam

base_model= VGG16(include_top=False, weights="imagenet", input_shape=(224,224,3))

#freezing
for layer in base_model.layers:
    layer.trainable= False

In [ ]:
model=Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dropout(0.6))
model.add(Dense(1024, activation="relu"))
model.add(Dropout(0.6))
model.add(Dense(1,activation="sigmoid"))

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history=model.fit(train_generator, epochs=15,validation_data=val_generator)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Transfer Learning - Accuracy over Epochs")
plt.legend()
plt.grid(True)
plt.show()

## Conclusion

In this project, a CNN-based image classification model was successfully developed to detect malaria-infected cells from microscopic images. The model achieved high accuracy on both training and validation datasets, demonstrating its ability to generalize well and distinguish between parasitized and healthy cells effectively.
The results suggest that deep learning can serve as a powerful tool in medical image analysis, particularly for assisting in rapid malaria diagnosis. Further improvements could include the use of transfer learning with pre-trained models, testing on larger and more diverse datasets, and eventually deploying the model in real-world mobile or edge applications for field use.

  Resources
- Chatgpt
- Course education /old projects
